In [84]:
from test.helpers import _batch_sde_solve, _batch_sde_solve_multi_y0

import diffrax
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.tree_util as jtu
from diffrax import LangevinTerm
from numpyro import diagnostics
from scipy import stats


jnp.set_printoptions(precision=4, suppress=True)
jax.config.update("jax_enable_x64", True)


def log_p_normal(x):
    return -0.5 * jnp.dot(x, x)


def log_p_neals_funnel(x):
    z_term = x[0] ** 2 / 6.0
    y_term = jnp.sum(x[1:] ** 2) / jax.lax.stop_gradient(2.0 * jnp.exp(x[0] / 2.0))
    return z_term + y_term


def p_neals_funnel(x):
    denom = jnp.sqrt((2.0 * jnp.pi) ** 10 * 3.0 * jnp.exp(9.0 * x[0] / 2.0))
    return jnp.exp(-log_p_neals_funnel(x)) / denom


@jax.jit
@jax.vmap
def neals_funnel_true_samples(keys):
    key_z, key_y = jr.split(keys, 2)
    z = jr.normal(key_z, shape=(1,)) * jnp.sqrt(3.0)
    y = jr.normal(key_y, shape=(9,)) * jnp.sqrt(jnp.exp(z / 2))
    return jnp.concatenate([z, y])


def rescale_neals_funnel(x):
    return jnp.concatenate(
        [x[:1] / jnp.sqrt(3.0), x[1:] / jnp.sqrt(jnp.exp(x[0] / 2.0))]
    )


def apply_f(f, samples):
    vf = jax.vmap(f, in_axes=0)
    return vf(samples)


def run_mcmc(
    key,
    log_p,
    x0,
    num_particles: int,
    chain_len: int,
    chain_sep: float = 0.1,
    tol: float = 1e-4,
):
    key_warmup, key_mcmc = jr.split(key, 2)
    keys_warmup = jr.split(key_warmup, num_particles)
    keys_mcmc = jr.split(key_mcmc, num_particles)
    grad_f = jax.grad(log_p)
    v0 = jnp.zeros_like(x0)
    y0 = (x0, v0)
    w_shape: tuple[int, ...] = x0.shape

    gamma, u = 1.0, 1.0

    def get_terms(bm):
        args = (gamma, u, grad_f)
        return LangevinTerm(args, bm)

    t_warmup = 16.0 * chain_sep
    controller_warmup = diffrax.PIDController(
        rtol=0.0, atol=8.0 * tol, pcoeff=0.1, icoeff=0.3, dtmin=2**-6
    )
    half_solver = diffrax.HalfSolver(diffrax.SORT(0.1))

    out_warmup, steps_warmup = _batch_sde_solve(
        keys_warmup,
        get_terms,
        w_shape,
        0.0,
        t_warmup,
        y0,
        None,
        half_solver,
        "space-time-time",
        0.1,
        controller_warmup,
        2**-9,
        diffrax.SaveAt(t1=True),
    )
    y_warm = jtu.tree_map(lambda x: jnp.squeeze(x, axis=1), out_warmup)

    t_mcmc = (chain_len + 4) * chain_sep
    save_ts = jnp.linspace(1.0, t_mcmc, num=chain_len, endpoint=True)
    saveat = diffrax.SaveAt(ts=save_ts)
    controller_mcmc = diffrax.PIDController(
        rtol=0.0, atol=tol, pcoeff=0.1, icoeff=0.3, dtmin=2**-9, step_ts=save_ts
    )
    out_mcmc, steps_mcmc = _batch_sde_solve_multi_y0(
        keys_mcmc,
        get_terms,
        w_shape,
        0.0,
        t_mcmc,
        y_warm,
        None,
        half_solver,
        "space-time-time",
        chain_sep,
        controller_mcmc,
        2**-12,
        saveat,
    )

    avg_steps_warmup = jnp.mean(steps_warmup)
    avg_steps_mcmc = jnp.mean(steps_mcmc)
    steps_per_sample = (avg_steps_mcmc + avg_steps_warmup) / chain_len
    print(
        f"Steps warmup: {avg_steps_warmup}, steps mcmc: {avg_steps_mcmc},"
        f" steps per output: {steps_per_sample}"
    )

    return out_mcmc[0], steps_per_sample


def evaluate_funnel(samples, steps_per_sample: float = 0.0):
    samples_rshp = jnp.reshape(samples, (-1, 10))
    means = jnp.mean(samples_rshp, axis=0)
    max_mean_err = jnp.max(jnp.abs(means))
    avg_mean_err = jnp.mean(jnp.abs(means))
    samples_rescaled = apply_f(rescale_neals_funnel, samples_rshp)
    cov = jnp.cov(samples_rescaled, rowvar=False)
    cov_err = jnp.abs(cov - jnp.eye(10))
    max_cov_err = jnp.max(cov_err)
    avg_cov_err = jnp.mean(cov_err)
    ref_dist = stats.norm(loc=0.0, scale=1.0)
    pvals = []
    for i in range(10):
        _, pval = stats.kstest(samples_rescaled[:, i], ref_dist.cdf)
        pvals.append(pval)
    min_pval = min(pvals)
    avg_pval = sum(pvals) / len(pvals)
    ess = diagnostics.effective_sample_size(ys_funnel_mcmc) / samples_rshp.shape[0]
    min_ess = jnp.min(ess)
    avg_ess = jnp.mean(ess)
    # compute the number of gradient evaluations per effective sample
    if steps_per_sample > 0:
        gepes = 6 * steps_per_sample / avg_ess
        print(f"Gradient evaluations per effective sample: {gepes}")
    print(
        f"Max mean err: {max_mean_err:.4}, max cov err: {max_cov_err:.4},"
        f" min_pval: {min_pval:.4}, min_ess {min_ess:.4}"
    )
    print(
        f"Avg mean err: {avg_mean_err:.4}, avg cov err: {avg_cov_err:.4},"
        f" avg_pval: {avg_pval:.4}, avg_ess {avg_ess:.4}"
    )
    return means, cov, pvals, ess

In [99]:
x0 = jnp.zeros((10,), dtype=jnp.float64)
ys_funnel_mcmc, sps_mcmc = run_mcmc(
    jr.PRNGKey(0), log_p_neals_funnel, x0, 64, 2**7, 2.0, 2.0**-6
)
print(ys_funnel_mcmc.shape)

Steps warmup: 102.90625, steps mcmc: 1707.296875, steps per output: 14.1422119140625
(64, 128, 10)


In [98]:
# 2**-4
means, cov, pvals, ess = evaluate_funnel(ys_funnel_mcmc, sps_mcmc)

Gradient evaluations per effective sample: 31.497757198799114
Max mean err: 0.0372, max cov err: 0.07994, min_pval: 0.07983, min_ess 0.3336
Avg mean err: 0.01655, avg cov err: 0.01696, avg_pval: 0.3455, avg_ess 0.5681


In [94]:
# 2**-6
means, cov, pvals, ess = evaluate_funnel(ys_funnel_mcmc, sps_mcmc)

Gradient evaluations per effective sample: 43.334091142582174
Max mean err: 0.03721, max cov err: 0.08164, min_pval: 0.07645, min_ess 0.3335
Avg mean err: 0.01658, avg cov err: 0.01712, avg_pval: 0.35, avg_ess 0.5681


In [92]:
# 2**-8
means, cov, pvals, ess = evaluate_funnel(ys_funnel_mcmc, sps_mcmc)

Gradient evaluations per effective sample: 57.86088252367586
Max mean err: 0.03723, max cov err: 0.08174, min_pval: 0.07885, min_ess 0.3335
Avg mean err: 0.01659, avg cov err: 0.01714, avg_pval: 0.3499, avg_ess 0.568


In [96]:
# 2**-10
means, cov, pvals, ess = evaluate_funnel(ys_funnel_mcmc, sps_mcmc)

Gradient evaluations per effective sample: 77.57345995483851
Max mean err: 0.03723, max cov err: 0.08175, min_pval: 0.08059, min_ess 0.3334
Avg mean err: 0.01659, avg cov err: 0.01714, avg_pval: 0.3491, avg_ess 0.568


In [100]:
# 2**-14
means, cov, pvals, ess = evaluate_funnel(ys_funnel_mcmc, sps_mcmc)

Gradient evaluations per effective sample: 149.39417148698837
Max mean err: 0.03723, max cov err: 0.08175, min_pval: 0.08233, min_ess 0.3334
Avg mean err: 0.01659, avg cov err: 0.01714, avg_pval: 0.3479, avg_ess 0.568


In [81]:
# True samples
keys = jr.split(jr.PRNGKey(0), 2**14)
ys_funnel_true = neals_funnel_true_samples(keys)
evaluate_funnel(ys_funnel_true)

Max mean err: 0.02297, max cov err: 0.03484, min_pval: 0.1016, min_ess 1.328
Avg mean err: 0.008169, avg cov err: 0.005785, avg_pval: 0.3735, avg_ess 1.801


(Array([-0.023 , -0.0118,  0.0012,  0.0066,  0.0102, -0.007 ,  0.0027,
        -0.014 ,  0.0042,  0.0012], dtype=float64),
 Array([[ 1.0164,  0.0036,  0.0156,  0.0034, -0.0022,  0.006 ,  0.0076,
         -0.0056,  0.0041,  0.0038],
        [ 0.0036,  0.9925,  0.002 ,  0.0129, -0.0044,  0.0067, -0.0062,
          0.0063,  0.0001, -0.0056],
        [ 0.0156,  0.002 ,  0.9652,  0.0028,  0.0001,  0.0018,  0.0034,
         -0.0015, -0.0013, -0.0048],
        [ 0.0034,  0.0129,  0.0028,  0.9841, -0.0005,  0.0078,  0.0001,
          0.0161,  0.0014,  0.0018],
        [-0.0022, -0.0044,  0.0001, -0.0005,  1.0003, -0.0067,  0.0092,
         -0.0064, -0.0048,  0.0085],
        [ 0.006 ,  0.0067,  0.0018,  0.0078, -0.0067,  1.005 ,  0.0006,
          0.0026,  0.0073,  0.0002],
        [ 0.0076, -0.0062,  0.0034,  0.0001,  0.0092,  0.0006,  1.0195,
         -0.0026,  0.0074,  0.0111],
        [-0.0056,  0.0063, -0.0015,  0.0161, -0.0064,  0.0026, -0.0026,
          0.9907,  0.0069, -0.0115],
     